In [ ]:
pip install weaviate-client langchain tiktoken pypdf rapidocr-onnxruntime sentence-transformers weviate

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [5]:
import os
import weaviate

from dotenv import load_dotenv
from weaviate.classes.init import Auth

In [14]:
load_dotenv(override=True)

weaviate_url = os.getenv("WEAVIATE_URL").strip()
weaviate_api_key = os.getenv("WEAVIATE_API_KEY").strip()

if not weaviate_url or not weaviate_api_key:
    raise ValueError("Weaviate credentials are missing from .env")
print("Endpoint:", weaviate_url)
print("Key loaded:", bool(weaviate_api_key))
print("Key length:", len(weaviate_api_key))

Endpoint: arq7h91mqjkyj8jqtycrq.c0.eu-central-1.aws.weaviate.cloud
Key loaded: True
Key length: 88


In [15]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=Auth.api_key(weaviate_api_key)
)

print("Connected:", client.is_ready())

Connected: True


In [19]:
import torch

print(torch.cuda.is_available())

False


In [20]:
# specify embedding model (using huggingface sentence transformer)
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
#model_kwargs = {"device": "cuda"}
embeddings = HuggingFaceEmbeddings(
  model_name=embedding_model_name,
  #model_kwargs=model_kwargs
)

c:\Users\LOQ\anaconda3\envs\Rag_Env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\LOQ\anaconda3\envs\Rag_Env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LOQ\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order 

In [22]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Book.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} pages")

C:\Users\LOQ\AppData\Local\Temp\ipykernel_11356\3742581129.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 300 pages


In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
docs = text_splitter.split_documents(pages)

In [32]:
from langchain_weaviate import WeaviateVectorStore

vector_db = WeaviateVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    client=client,
    index_name="RagDocuments",
    text_key="text"
)

simsimd not available — falling back to SciPy/NumPy implementation for vector math. To enable the accelerated path, install the optional dependency 'simsimd' (note: simsimd may require newer glibc on some systems).
